# IJGE / Energy Sources A revision — Stage A (read-only export + reproduction checks)

**What this notebook does**
- Reads your frozen V6 artifacts on Google Drive. **Nothing is trained, nothing is re-inferred, no frozen file is modified.**
- Writes only to `Benchmark_V5/IJGE_Revision/StageA_export/`.
- Builds one consolidated prediction file (all models × horizons × seeds, in MW), reproduces the manuscript's Tables 5, 6 and 9 from it, and packs small CSV/JSON artifacts so the remaining analyses can be done offline.

**Runtime:** CPU is enough (≈5–10 min, mostly Drive I/O). Run cells A0 → A4 in order and send me `StageA_bundle.zip` plus the printed output of A1 and A3.

In [ ]:
# ===================== A0 — SETUP (read-only roots, single write location) =====================
from google.colab import drive
drive.mount('/content/gdrive')

import os, sys, json, hashlib, platform, shutil, zipfile, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd

MYDRIVE = Path('/content/gdrive/MyDrive')
BENCH   = MYDRIVE / 'Benchmark_V5'
OUT     = BENCH / 'IJGE_Revision' / 'StageA_export'
OUT.mkdir(parents=True, exist_ok=True)

SEEDS    = [1, 7, 21, 42, 100]
N_TEST   = 6576
# Training-set target scaler (identical to the constants frozen in Step 9A / 12J-9)
TARGET_MEAN  = np.array([55821.81814021338, 4184.239495437773])
TARGET_SCALE = np.array([10034.618508055319, 6415.360950933769])
EXPECTED_1H_SHA = 'ed67feaed54517cac5f84d314caca75a7b3837ed9bb7f17c7613f0023bd69912'

def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def find_dir(name):
    """Locate a frozen experiment folder without guessing: Benchmark_V5 first, then MyDrive root."""
    for c in [BENCH / name, MYDRIVE / name]:
        if c.is_dir():
            return c
    hits = [p for p in BENCH.rglob(name) if p.is_dir()]
    assert len(hits) == 1, f'Could not uniquely locate {name}: {hits}'
    return hits[0]

def guard_write(p):
    p = Path(p).resolve()
    assert str(p).startswith(str(OUT.resolve())), f'REFUSING to write outside OUT: {p}'
    return p

print('Python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__)
print('Benchmark_V5 exists:', BENCH.exists())
print('Write location     :', OUT)


In [ ]:
# ===================== A1 — INVENTORY (read-only; prints what will be exported) =====================
DATA_1H = BENCH / 'data' / 'frozen_168h_sequences.npz'
TF_1H   = BENCH / 'Reliability_V6/degradation_results/tensorflow_models/tensorflow_reliability_clean_predictions.npz'
PT_1H   = BENCH / 'Reliability_V6/degradation_results/pytorch_models/pytorch_reliability_clean_predictions.npz'
MH_PRED = BENCH / 'MultiHorizon_V6/definitive_training/predictions'
MTLGAP1 = find_dir('MTL_GAP_vs_LSTA_V6')
ESWA    = find_dir('ESWA_Revision')

for p in [DATA_1H, TF_1H, PT_1H, MH_PRED, MTLGAP1, ESWA]:
    print(('OK   ' if p.exists() else 'MISS ') + str(p))

h = sha256(DATA_1H)
print('\n1-h dataset SHA256:', h, '| PASS' if h == EXPECTED_1H_SHA else '| *** MISMATCH ***')
assert h == EXPECTED_1H_SHA

def show_keys(p):
    with np.load(p, allow_pickle=False) as z:
        print(f'  {p.relative_to(MYDRIVE)}')
        for k in z.files[:12]:
            print(f'     {k:40s} {z[k].shape} {z[k].dtype}')
        if len(z.files) > 12: print(f'     ... {len(z.files)} keys total')

print('\nKey layouts:')
for p in [TF_1H, PT_1H,
          MTLGAP1 / 'predictions' / 'MTL_GAP_seed_1_predictions.npz',
          MH_PRED / 'h06_seed001_ST_Load_predictions.npz',
          MH_PRED / 'h06_seed001_MTL_LSTA_predictions.npz',
          ESWA / 'Correction_2_MTL_GAP' / 'h06_seed001' / 'predictions.npz']:
    show_keys(p)

st_lsta_dirs = sorted((ESWA / 'Correction_2_ST_LSTA').glob('*h01_seed*'))
print('\n1-h ST-LSTA run folders:', [d.name for d in st_lsta_dirs])

print('\nNPZ files under Reliability_V6 (to see whether degraded predictions were saved):')
for p in sorted((BENCH / 'Reliability_V6').rglob('*.npz')):
    print(f'  {p.stat().st_size/1e6:8.2f} MB  {p.relative_to(BENCH)}')

naive = sorted(BENCH.rglob('*naive*predictions*.npz'))
print('\nNaive-baseline prediction files:', [str(p.relative_to(BENCH)) for p in naive])


In [ ]:
# ===================== A2 — CONSOLIDATED PREDICTION EXPORT (MW) =====================
bundle = {}
log = []

# ---- 1-h ground truth + dynamic history (needed offline for condition-matched baselines) ----
with np.load(DATA_1H, allow_pickle=False) as z:
    X_test = np.asarray(z['X_test'], dtype=np.float64)
    y_std  = np.asarray(z['y_test'], dtype=np.float64)
assert X_test.shape == (N_TEST, 168, 8) and y_std.shape == (N_TEST, 2)
y1 = y_std * TARGET_SCALE + TARGET_MEAN
X_dyn_mw = (X_test[:, :, :2] * TARGET_SCALE + TARGET_MEAN).astype(np.float32)   # load, solar history in MW
bundle['y_true_h1']   = y1
bundle['X_dyn_h1_MW'] = X_dyn_mw

def put(h, model, seed, pred, truth=None, tol=0.05):
    pred = np.asarray(pred, dtype=np.float64)
    assert pred.shape == (N_TEST, 2) and np.isfinite(pred).all(), (h, model, seed, pred.shape)
    if truth is not None:
        d = float(np.max(np.abs(np.asarray(truth, dtype=np.float64) - bundle[f'y_true_h{h}'])))
        log.append((h, model, seed, d))
        assert d < tol, f'truth mismatch {h} {model} {seed}: {d}'
    bundle[f'h{h}__{model}__s{seed}'] = pred

# ---- 1-h five-model benchmark (authoritative clean bundles used in Steps 8–9) ----
tf = np.load(TF_1H, allow_pickle=False); pt = np.load(PT_1H, allow_pickle=False)
for s in SEEDS:
    put(1, 'LSTM',     s, tf[f'LSTM_seed_{s}_pred_mw'],           tf['y_true_mw'])
    put(1, 'ST-GAP',   s, tf[f'ST_Transformer_seed_{s}_pred_mw'], tf['y_true_mw'])
    put(1, 'MTL-LSTA', s, tf[f'MTL_LSTA_seed_{s}_pred_mw'],       tf['y_true_mw'])
    put(1, 'PatchTST', s, pt[f'PatchTST_seed_{s}_pred_mw'],       pt['y_true_mw'])
    put(1, 'DLinear',  s, pt[f'DLinear_seed_{s}_pred_mw'],        pt['y_true_mw'])

# ---- 1-h factorial arms ----
def st_lsta_dir(hz, s):
    root = ESWA / 'Correction_2_ST_LSTA'
    exact = root / f'h{hz:02d}_seed{s:03d}'
    if exact.is_dir(): return exact
    hits = sorted(root.glob(f'*h{hz:02d}_seed{s:03d}'))
    assert len(hits) == 1, (hz, s, hits)
    return hits[0]

for s in SEEDS:
    d = st_lsta_dir(1, s)
    zl = np.load(d / 'ST_LSTA_load_predictions.npz'); zs = np.load(d / 'ST_LSTA_solar_predictions.npz')
    put(1, 'ST-LSTA', s, np.c_[zl['y_pred_mw'], zs['y_pred_mw']], np.c_[zl['y_true_mw'], zs['y_true_mw']])

    for arm in ['GAP', 'LSTA']:
        z = np.load(MTLGAP1 / 'predictions' / f'MTL_{arm}_seed_{s}_predictions.npz')
        pred = np.c_[z['y_pred_load'], z['y_pred_solar']].astype(np.float64) * TARGET_SCALE + TARGET_MEAN
        tru  = np.c_[z['y_true_load'], z['y_true_solar']].astype(np.float64) * TARGET_SCALE + TARGET_MEAN
        put(1, f'MTL-{arm}' if arm == 'GAP' else 'MTL-LSTA_factorialfile', s, pred, tru)

# consistency: MTL-LSTA in benchmark bundle vs MTL_GAP_vs_LSTA_V6 file (should be the same runs)
for s in SEEDS:
    d = np.max(np.abs(bundle[f'h1__MTL-LSTA__s{s}'] - bundle[f'h1__MTL-LSTA_factorialfile__s{s}']))
    print(f'1-h MTL-LSTA benchmark vs factorial file, seed {s:3d}: max |diff| = {d:.6f} MW')

# ---- 6-h and 24-h ----
def mh(hz, s, phys):
    z = np.load(MH_PRED / f'h{hz:02d}_seed{s:03d}_{phys}_predictions.npz', allow_pickle=False)
    return np.asarray(z['y_true_physical'], np.float64), np.asarray(z['y_pred_physical'], np.float64), z

for hz in [6, 24]:
    t0, _, _ = mh(hz, 1, 'LSTM'); bundle[f'y_true_h{hz}'] = t0
    for s in SEEDS:
        for name, phys in [('LSTM', 'LSTM'), ('MTL-LSTA', 'MTL_LSTA'), ('PatchTST', 'PatchTST'), ('DLinear', 'DLinear')]:
            t, p, _ = mh(hz, s, phys); put(hz, name, s, p, t)
        tl, pl, _ = mh(hz, s, 'ST_Load'); ts_, ps, _ = mh(hz, s, 'ST_Solar')
        sel = lambda a, i: a[:, i] if a.ndim == 2 else a.reshape(-1)
        put(hz, 'ST-GAP', s, np.c_[sel(pl, 0), sel(ps, 1)], np.c_[sel(tl, 0), sel(ts_, 1)])
        d = st_lsta_dir(hz, s)
        zl = np.load(d / 'ST_LSTA_load_predictions.npz'); zs = np.load(d / 'ST_LSTA_solar_predictions.npz')
        put(hz, 'ST-LSTA', s, np.c_[zl['y_pred_mw'], zs['y_pred_mw']], np.c_[zl['y_true_mw'], zs['y_true_mw']])
        # MTL-GAP: stored standardized only -> exact affine map certified on the paired MTL-LSTA truth
        zg = np.load(ESWA / 'Correction_2_MTL_GAP' / f'h{hz:02d}_seed{s:03d}' / 'predictions.npz')
        _, _, zm = mh(hz, s, 'MTL_LSTA')
        assert np.max(np.abs(zg['y_true_scaled'] - zm['y_true_scaled'])) < 1e-4
        pred = np.empty((N_TEST, 2))
        for i in range(2):
            a, b = np.polyfit(np.asarray(zm['y_true_scaled'][:, i], np.float64), np.asarray(zm['y_true_physical'][:, i], np.float64), 1)
            resid = np.max(np.abs(a * zm['y_true_scaled'][:, i] + b - zm['y_true_physical'][:, i]))
            assert resid < 0.05, ('affine residual', hz, s, i, resid)
            pred[:, i] = a * np.asarray(zg['y_pred_scaled'][:, i], np.float64) + b
        put(hz, 'MTL-GAP', s, pred)

# ---- cross-horizon target alignment (are the 6,576 test targets identical across horizons?) ----
for hz in [6, 24]:
    print(f'Truth h{hz} vs h1: max |diff| = {np.max(np.abs(bundle[f"y_true_h{hz}"] - y1)):.6f} MW')

dest = guard_write(OUT / 'predictions_all_MW.npz')
np.savez_compressed(dest, **bundle)
print(f'\nSaved {len(bundle)} arrays -> {dest} ({dest.stat().st_size/1e6:.1f} MB)')
print('Max truth discrepancy across all loaded files: %.6f MW' % max(x[3] for x in log))


In [ ]:
# ===================== A3 — REPRODUCTION CHECKS vs MANUSCRIPT TABLES 5, 6, 9 =====================
# Published values (Final_Article V7). RMSE mean ± sample SD over five seeds.
T5 = {  # (h, target): {model: (mean, sd)}
 (1,'Load'):  {'LSTM':(818.183,22.641),'ST-GAP':(786.593,70.289),'MTL-LSTA':(708.822,48.926),'PatchTST':(697.373,11.331),'DLinear':(760.031,2.842)},
 (1,'Solar'): {'LSTM':(469.731,24.015),'ST-GAP':(524.389,75.131),'MTL-LSTA':(541.201,50.358),'PatchTST':(540.160,11.524),'DLinear':(489.307,5.068)},
 (6,'Load'):  {'LSTM':(2116.388,94.095),'ST-GAP':(1597.430,50.490),'MTL-LSTA':(1636.578,115.344),'PatchTST':(1549.840,11.473),'DLinear':(2447.955,6.707)},
 (6,'Solar'): {'LSTM':(2342.239,93.308),'ST-GAP':(2173.919,38.553),'MTL-LSTA':(2271.276,167.856),'PatchTST':(2003.036,6.037),'DLinear':(2621.256,3.075)},
 (24,'Load'): {'LSTM':(2923.848,282.076),'ST-GAP':(2788.721,289.077),'MTL-LSTA':(2827.920,146.863),'PatchTST':(2374.515,18.570),'DLinear':(3124.898,2.823)},
 (24,'Solar'):{'LSTM':(2824.578,62.648),'ST-GAP':(2455.216,35.492),'MTL-LSTA':(2620.557,61.362),'PatchTST':(2442.405,8.193),'DLinear':(2774.535,4.997)},
}
T9 = {
 (1,'Load'):{'ST-LSTA':(705.108,50.432),'MTL-GAP':(943.804,174.383)},  (1,'Solar'):{'ST-LSTA':(509.149,30.702),'MTL-GAP':(722.378,93.112)},
 (6,'Load'):{'ST-LSTA':(1497.036,67.460),'MTL-GAP':(2124.689,112.585)},(6,'Solar'):{'ST-LSTA':(2305.616,119.812),'MTL-GAP':(2247.609,38.176)},
 (24,'Load'):{'ST-LSTA':(2724.339,177.278),'MTL-GAP':(3226.088,387.435)},(24,'Solar'):{'ST-LSTA':(2449.564,31.193),'MTL-GAP':(2508.340,40.978)},
}
T6 = {(1,'Load'):(2391.092,6161.737,3777.580),(1,'Solar'):(2583.798,2633.191,4010.645),
      (6,'Load'):(10708.304,6161.737,3777.580),(6,'Solar'):(12149.543,2633.191,4010.645),
      (24,'Load'):(6161.737,6161.737,3777.580),(24,'Solar'):(2633.191,2633.191,4010.645)}
TI = {'Load':0,'Solar':1}
rmse = lambda e: float(np.sqrt(np.mean(e**2)))

rows = []
for table, ref in [('T5', T5), ('T9', T9)]:
    for (hz, tgt), models in ref.items():
        y = bundle[f'y_true_h{hz}'][:, TI[tgt]]
        for m, (pm, psd) in models.items():
            r = np.array([rmse(bundle[f'h{hz}__{m}__s{s}'][:, TI[tgt]] - y) for s in SEEDS])
            dm, dsd = r.mean() - pm, r.std(ddof=1) - psd
            status = 'PASS' if max(abs(dm), abs(dsd)) <= 0.0015 else ('PASS*' if max(abs(dm), abs(dsd)) <= 0.01 else 'FAIL')
            rows.append([table, hz, tgt, m, r.mean(), r.std(ddof=1), pm, psd, dm, dsd, status])

# Persistence baselines rebuilt from the 1-h input windows (target T; window ends at T-1)
X = bundle['X_dyn_h1_MW'].astype(np.float64)
for (hz, tgt), (op, dp, wp) in T6.items():
    i = TI[tgt]; y = bundle[f'y_true_h{hz}'][:, i]
    preds = {'Ordinary': X[:, -hz, i], 'Daily': X[:, -24, i], 'Weekly': X[:, -168, i]}
    for nm, ref_v in zip(['Ordinary', 'Daily', 'Weekly'], [op, dp, wp]):
        v = rmse(preds[nm] - y)
        rows.append(['T6', hz, tgt, nm + ' persistence', v, np.nan, ref_v, np.nan, v - ref_v, np.nan,
                     'PASS' if abs(v - ref_v) <= 0.0015 else ('PASS*' if abs(v - ref_v) <= 0.01 else 'FAIL')])

rep = pd.DataFrame(rows, columns=['Table','h','Target','Model','RMSE_mean','RMSE_sd','Paper_mean','Paper_sd','d_mean','d_sd','Status'])
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 200)
print(rep.round(4).to_string(index=False))
print('\nStatus counts:', rep['Status'].value_counts().to_dict())
print('PASS = within rounding (<=0.0015 MW); PASS* = <=0.01 MW (expected for 1-h ST, re-inferred checkpoint, documented 0.0078 MW drift); FAIL = investigate')
rep.to_csv(guard_write(OUT / 'reproduction_check_tables_5_6_9.csv'), index=False)


In [ ]:
# ===================== A4 — PACK SMALL ARTIFACTS + BUNDLE FOR DOWNLOAD =====================
ART = OUT / 'artifacts'; ART.mkdir(exist_ok=True)
SKIP_PARTS = {'checkpoints', '.ipynb_checkpoints', 'IJGE_Revision', 'checkpoint'}
copied, total = 0, 0
for p in BENCH.rglob('*'):
    if not p.is_file() or p.suffix.lower() not in {'.csv', '.json'}: continue
    if SKIP_PARTS & set(p.relative_to(BENCH).parts): continue
    sz = p.stat().st_size
    if sz > 2_000_000 or total + sz > 150_000_000: continue
    dst = guard_write(ART / p.relative_to(BENCH)); dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dst); copied += 1; total += sz
# random-degradation masks (needed for condition-matched persistence offline)
for p in (BENCH / 'Reliability_V6' / 'degradation_protocol').glob('*.npz'):
    dst = guard_write(ART / p.relative_to(BENCH)); dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(p, dst)
print(f'Copied {copied} CSV/JSON files ({total/1e6:.1f} MB) + mask files')

zpath = guard_write(OUT / 'StageA_bundle.zip')
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT / 'predictions_all_MW.npz', 'predictions_all_MW.npz')
    zf.write(OUT / 'reproduction_check_tables_5_6_9.csv', 'reproduction_check_tables_5_6_9.csv')
    for p in ART.rglob('*'):
        if p.is_file(): zf.write(p, str(p.relative_to(OUT)))
print(f'Bundle: {zpath}  ({zpath.stat().st_size/1e6:.1f} MB)')
print('Download it from Drive (Benchmark_V5/IJGE_Revision/StageA_export/) or run the next line:')
print("from google.colab import files; files.download(str(zpath))")
